# 01 — CRCP Benthic Cover Data Exploration

This notebook profiles the NOAA CRCP benthic cover dataset for Hawaii,
accessed from ERDDAP (`CRCP_Benthic_Cover_Hawaii`). It documents the
schema, taxonomy hierarchy, data distributions, and quality issues that
inform the design of the processing pipeline.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from erddapy import ERDDAP
from src.config import get_source

## 1. Connect to ERDDAP and fetch data

In [7]:
cfg = get_source("hawaii")
print(f"Dataset: {cfg.dataset_id}")
print(f"Server:  {cfg.erddap_server}")

e = ERDDAP(
    server=cfg.erddap_server,
    protocol="tabledap",
)
e.dataset_id = cfg.dataset_id
e.variables = [
    "latitude", "longitude", "region_name", "island", "site",
    "reef_zone", "depth_bin", "min_depth", "max_depth",
    "image_name", "image_url", "obs_year", "date_",
    "analyst", "tier_1", "category_name",
    "tier_2", "subcategory_name", "tier_3", "genera_name",
]

print("Fetching data from ERDDAP (this may take a minute)...")
df = e.to_pandas()
print(f"Rows fetched: {len(df):,}")

df.columns = [c.split(" (")[0] for c in df.columns]

df.head()

Dataset: CRCP_Benthic_Cover_Hawaii
Server:  https://www.ncei.noaa.gov/erddap
Fetching data from ERDDAP (this may take a minute)...
Rows fetched: 472,070


,latitude,longitude,region_name,island,site,reef_zone,depth_bin,min_depth,max_depth,image_name,image_url,obs_year,date_,analyst,tier_1,category_name,tier_2,subcategory_name,tier_3,genera_name
0,20.069002,-155.391537,Main Hawaiian Islands,Hawaii,HAW-3434,Forereef,Deep,68.0,72.0,HAW-3434_2019_A_01.JPG,https://accession.nodc.noaa.gov/0211063/data/0...,2019,2019-05-04T00:00:00Z,elooney12,SED,Sediment,SAND,Sand,SAND,Sand
1,20.069002,-155.391537,Main Hawaiian Islands,Hawaii,HAW-3434,Forereef,Deep,68.0,72.0,HAW-3434_2019_A_01.JPG,https://accession.nodc.noaa.gov/0211063/data/0...,2019,2019-05-04T00:00:00Z,elooney12,SED,Sediment,SAND,Sand,SAND,Sand
2,20.069002,-155.391537,Main Hawaiian Islands,Hawaii,HAW-3434,Forereef,Deep,68.0,72.0,HAW-3434_2019_A_01.JPG,https://accession.nodc.noaa.gov/0211063/data/0...,2019,2019-05-04T00:00:00Z,elooney12,SED,Sediment,SAND,Sand,SAND,Sand
3,20.069002,-155.391537,Main Hawaiian Islands,Hawaii,HAW-3434,Forereef,Deep,68.0,72.0,HAW-3434_2019_A_01.JPG,https://accession.nodc.noaa.gov/0211063/data/0...,2019,2019-05-04T00:00:00Z,elooney12,SED,Sediment,SAND,Sand,SAND,Sand
4,20.069002,-155.391537,Main Hawaiian Islands,Hawaii,HAW-3434,Forereef,Deep,68.0,72.0,HAW-3434_2019_A_01.JPG,https://accession.nodc.noaa.gov/0211063/data/0...,2019,2019-05-04T00:00:00Z,elooney12,MA,Macroalga,UPMA,Upright macroalga,PESP,Peyssonnelia sp


## 2. Schema overview

In [3]:
print(f"Shape: {df.shape}")
print(f"\nColumns and dtypes:")
for col in df.columns:
    non_null = df[col].notna().sum()
    print(f"  {col:30s}  {str(df[col].dtype):10s}  ({non_null:,} non-null)")

Shape: (472070, 20)

Columns and dtypes:
  latitude (degrees_north)        float64     (472,070 non-null)
  longitude (degrees_east)        float64     (472,070 non-null)
  region_name                     object      (472,070 non-null)
  island                          object      (472,070 non-null)
  site                            object      (472,070 non-null)
  reef_zone                       object      (472,070 non-null)
  depth_bin                       object      (472,070 non-null)
  min_depth (ft)                  float64     (159,790 non-null)
  max_depth (ft)                  float64     (161,010 non-null)
  image_name                      object      (472,070 non-null)
  image_url                       object      (472,070 non-null)
  obs_year (years)                int64       (472,070 non-null)
  date_ (UTC)                     object      (469,670 non-null)
  analyst                         object      (472,070 non-null)
  tier_1                          object      (47

## 3. Distributions

In [15]:
print("=== Survey Years ===")
print(df["obs_year"].value_counts().sort_index().to_string())

print(f"\n=== Islands ({df['island'].nunique()}) ===")
print(df["island"].value_counts().sort_index().to_string())

print(f"\n=== Reef Zones ===")
print(df["reef_zone"].value_counts().to_string())

print(f"\n=== Depth Bins ===")
print(df["depth_bin"].value_counts().to_string())

print(f"\nUnique sites:  {df['site'].nunique():,}")
print(f"Unique images: {df['image_name'].nunique():,}")
print(f"Unique coords: {df.groupby(['latitude','longitude']).ngroups:,}")

=== Survey Years ===
obs_year
2013    126280
2016    196670
2019    149120

=== Islands (12) ===
island
French Frigate    21750
Hawaii            87990
Kahoolawe         19590
Kauai             41190
Kure              17560
Lanai             39130
Lisianski         16250
Maui              46000
Molokai           46240
Niihau            26120
Oahu              88280
Pearl & Hermes    21970

=== Reef Zones ===
reef_zone
Forereef           461460
Protected Slope      5490
Lagoon               5120

=== Depth Bins ===
depth_bin
Mid        205200
Shallow    138130
Deep       128740

Unique sites:  1,553
Unique images: 46,727
Unique coords: 1,546


## 4. Tier taxonomy tree

In [9]:
tier_map = defaultdict(lambda: defaultdict(set))
for _, row in df[["tier_1", "category_name", "tier_2", "subcategory_name", "tier_3", "genera_name"]].drop_duplicates().iterrows():
    t1 = f"{row['tier_1']} ({row['category_name']})"
    t2 = f"{row['tier_2']} ({row['subcategory_name']})"
    t3 = f"{row['tier_3']} ({row['genera_name']})" if pd.notna(row['tier_3']) and row['tier_3'] != '' else None
    if t3:
        tier_map[t1][t2].add(t3)
    else:
        tier_map[t1][t2]  # ensure key exists

for t1 in sorted(tier_map):
    print(f"\n{t1}")
    for t2 in sorted(tier_map[t1]):
        children = sorted(tier_map[t1][t2])
        print(f"  └─ {t2}")
        for t3 in children:
            print(f"       └─ {t3}")


CCA (Coralline Alga)
  └─ CCAH (CCA growing on hard substrate)
       └─ CCAH (CCA growing on hard substrate)
  └─ CCAR (CCA growing on rubble)
       └─ CCAR (CCA growing on rubble)

CORAL (Coral)
  └─ BR (Branching hard coral)
       └─ ACBR (Acropora branching)
       └─ BR (Branching hard coral)
       └─ MOBR (Montipora branching)
       └─ POBR (Porites branching)
       └─ POCS (Pocillopora sp)
       └─ PONM (Porites non-massive)
  └─ COL (Columnar hard coral)
       └─ COL (Columnar hard coral)
  └─ ENC (Encrusting hard coral)
       └─ ACAS (Acanthastrea sp)
       └─ COSP (Coscinaraea sp)
       └─ CYPS (Cyphastrea sp)
       └─ ENC (Encrusting hard coral)
       └─ LEPT (Leptastrea sp)
       └─ MOEN (Montipora encrusting)
       └─ PAEN (Pavona encrusting)
       └─ PAVS (Pavona sp)
       └─ POEN (Porites encrusting)
       └─ PSSP (Psammocora sp)
  └─ FOL (Foliose hard coral)
       └─ ECHP (Echinopora sp)
       └─ FOL (Foliose hard coral)
       └─ MESP (Merulina sp)


## 5. Points per image

In [10]:
pts_per_img = df.groupby("image_name").size()
print(f"Points per image — min: {pts_per_img.min()}, max: {pts_per_img.max()}, "
      f"median: {pts_per_img.median()}, mean: {pts_per_img.mean():.1f}")
print(f"\nDistribution:")
print(pts_per_img.value_counts().sort_index().to_string())

Points per image — min: 10, max: 30, median: 10.0, mean: 10.1

Distribution:
10    46487
30      240


## 6. Multi-year site coverage

In [11]:
site_years = df.groupby("site")["obs_year"].apply(lambda s: tuple(sorted(s.unique())))
multi = site_years[site_years.apply(len) > 1]
print(f"Sites surveyed in 1 year only: {(site_years.apply(len) == 1).sum()}")
print(f"Sites surveyed in 2+ years:    {len(multi)}")
if len(multi) > 0:
    print("\nYear combinations:")
    print(multi.value_counts().to_string())

Sites surveyed in 1 year only: 1553
Sites surveyed in 2+ years:    0


## 7. Tape/Wand and Unclassified prevalence

In [12]:
tier1_counts = df["tier_1"].value_counts()
total = len(df)
for label in ["TW", "UC"]:
    n = tier1_counts.get(label, 0)
    print(f"{label}: {n:,} points ({100*n/total:.2f}%)")

tw_per_img = df[df["tier_1"] == "TW"].groupby("image_name").size()
print(f"\nImages containing TW points: {len(tw_per_img):,} / {df['image_name'].nunique():,}")
if len(tw_per_img) > 0:
    print(f"TW points per affected image — min: {tw_per_img.min()}, max: {tw_per_img.max()}, median: {tw_per_img.median()}")

TW: 1,567 points (0.33%)
UC: 9,102 points (1.93%)

Images containing TW points: 1,512 / 46,727
TW points per affected image — min: 1, max: 3, median: 1.0


## 8. Coordinate quality check

In [13]:
null_lat = df["latitude"].isna().sum()
null_lon = df["longitude"].isna().sum()
print(f"Null latitude:  {null_lat}")
print(f"Null longitude: {null_lon}")

print(f"\nLatitude  range: {df['latitude'].min():.6f} to {df['latitude'].max():.6f}")
print(f"Longitude range: {df['longitude'].min():.6f} to {df['longitude'].max():.6f}")

# Check for sites with multiple coordinate sets
site_coords = df.groupby("site").agg(
    n_coords=("latitude", lambda s: len(set(zip(s, df.loc[s.index, "longitude"]))))
)
multi_coord = site_coords[site_coords["n_coords"] > 1]
print(f"\nSites with single coordinate: {(site_coords['n_coords'] == 1).sum()}")
print(f"Sites with multiple coordinates: {len(multi_coord)}")

Null latitude:  0
Null longitude: 0

Latitude  range: 18.919225 to 28.457971
Longitude range: -178.384330 to -154.804172

Sites with single coordinate: 1553
Sites with multiple coordinates: 0


## 9. Survey date consistency

Check whether each site was surveyed on a single date, which would let us
use the actual sampling date instead of a synthetic mid-year placeholder.

In [ ]:
null_dates = df["date_"].isna().sum()
print(f"Null date_ values: {null_dates:,} / {len(df):,}")

dates_per_site = df.groupby("site")["date_"].nunique()
print(f"\nSites with 1 date:   {(dates_per_site == 1).sum()}")
print(f"Sites with 2+ dates: {(dates_per_site > 1).sum()}")

if (dates_per_site > 1).any():
    print("\nMulti-date sites:")
    multi = dates_per_site[dates_per_site > 1]
    for site in multi.index[:10]:
        site_dates = df[df["site"] == site]["date_"].unique()
        print(f"  {site}: {sorted(site_dates)}")